# Exploratory Data Analysis (EDA)

## Overview
Understand the structure, balance, and quality of the flight delay data. 
Identify missingness patterns, verify target distribution, and confirm 
leakage detection.

### Importing os

This step makes sure the project directory is connected to the notebooks

In [1]:
import os
os.chdir("..")  # Go up one level from notebooks/ to project root
print(os.getcwd())

C:\Users\hanis\Flight\flight-delay-prediction


Here we can look at shape of the dataset
The data types of all the columns
The first 5 rows for reference

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load merged data
df = pd.read_parquet("data/processed/flights.parquet")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst few rows:")
df.head()

Shape: (2983129, 109)

Data types:
Year                  int64
Quarter               int64
Month                 int64
DayofMonth            int64
DayOfWeek             int64
                     ...   
Div5WheelsOn        float64
Div5TotalGTime      float64
Div5LongestGTime    float64
Div5WheelsOff       float64
Div5TailNum         float64
Length: 109, dtype: object

First few rows:


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,2024,4,10,29,2,2024-10-29,UA,19977,UA,N37504,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,4,10,29,2,2024-10-29,UA,19977,UA,N69824,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,4,10,29,2,2024-10-29,UA,19977,UA,N37297,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,4,10,29,2,2024-10-29,UA,19977,UA,N47569,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024,4,10,29,2,2024-10-29,UA,19977,UA,N69826,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Checking if the data types are right

The datatype for "FlightDate" is string. We need that in datetime

In [3]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
print(df["FlightDate"].dtype)

datetime64[ns]


## Target Balance

The dataset is imbalanced: 82% on-time flights, 18% delayed >15 minutes. 
This imbalance means accuracy alone is misleading—we'll use PR-AUC and 
precision/recall as primary metrics.

In [4]:
# Target balance
print("Target Distribution (DepDel15):")
print(df['DepDel15'].value_counts())
print("\nTarget balance (%):")
print(df['DepDel15'].value_counts(normalize=True) * 100)

Target Distribution (DepDel15):
DepDel15
0.0    2435164
1.0     519811
Name: count, dtype: int64

Target balance (%):
DepDel15
0.0    82.408954
1.0    17.591046
Name: proportion, dtype: float64


## Missing values

Some columns have 100% missing values. Hence these columns can be dropped later on
The columns whose missing value percentages are small need to be handled.

In [5]:
print("\nMISSING VALUES")
print("="*50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing[missing > 0],
    'Percentage': missing_pct[missing > 0]
}).sort_values('Missing_Count', ascending=False)
print(missing_df)


MISSING VALUES
                                 Missing_Count  Percentage
Div5TailNum                            2983129  100.000000
Div4WheelsOn                           2983129  100.000000
Div3Airport                            2983129  100.000000
Div3AirportID                          2983129  100.000000
Div3AirportSeqID                       2983129  100.000000
...                                        ...         ...
DepDelayMinutes                          28154    0.943774
DepDelay                                 28154    0.943774
DepTime                                  28071    0.940992
Tail_Number                               6302    0.211255
Flight_Number_Reporting_Airline              1    0.000034

[71 rows x 2 columns]


## Leakage sanity check

DepDelay (0.5462): Surprisingly not 1.0, but here's why—correlation measures linear relationship. DepDel15 is a binary threshold (DepDelay > 15), not linear. So 0.5462 is actually strong and confirms DepDelay is highly predictive (but not perfectly, due to the threshold nature).

ArrDelay (0.5312): Similar—still leakage because it's arrival delay (post-departure event), but the correlation is moderate due to the threshold.

TaxiOut & ActualElapsedTime (0.02–0.06): Surprisingly low. This tells you something: leakage ≠ high correlation. These columns are information from the future (post-departure), but they don't happen to correlate strongly with the target. Still leakage though, because they're unavailable at prediction time.

Key lesson: Don't rely on correlation alone to detect leakage. Rely on your decision-time boundary (2 hours before scheduled departure). If information only becomes available at/after that time, it's leakage—regardless of correlation.

In [6]:
# Quick correlation check: leaky features should correlate ~1.0 with target
leaky_cols = ['DepDelay', 'ArrDelay', 'ActualElapsedTime', 'TaxiOut']
print("LEAKAGE SANITY CHECK")
print("="*50)
print("Correlation of leaky features with target (DepDel15):")
for col in leaky_cols:
    if col in df.columns:
        corr = df[col].corr(df['DepDel15'])
        print(f"{col:30s}: {corr:.4f}")

LEAKAGE SANITY CHECK
Correlation of leaky features with target (DepDel15):
DepDelay                      : 0.5462
ArrDelay                      : 0.5312
ActualElapsedTime             : 0.0232
TaxiOut                       : 0.0567


# Checking duplicate values
- Checked for duplicate records using df.duplicated().sum().
- No duplicate rows were found.
- No duplicate removal is required.

In [10]:
print("The no. of duplicate rows are:");
print(df.duplicated().sum())

The no. of duplicate rows are:
0


# Checking for Impossible values

### 1. CRSDepTime cannot have 2400
    In our case, we got all valid.

In [13]:
print("Max CRSDepTime:", df["CRSDepTime"].max())
print("Rows with 2400:", (df["CRSDepTime"] == 2400).sum())

Max CRSDepTime: 2359
Rows with 2400: 0


if present convert to 0(midnight)

### 2. Check Distance: if it is less than 0, then invalid
    In our case, we got all valid distances.

In [14]:
print("Minimum Distance:", df["Distance"].min())
print("Rows with non-positive distance:",
      (df["Distance"] <= 0).sum())

Minimum Distance: 18.0
Rows with non-positive distance: 0
